In [ ]:
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loaders.mice import MiceDatasetLoader
from aind_analysis_arch_result_access import han_pipeline

# pd.set_option('display.max_rows', None)

## merge data df

### load session level meta data

In [ ]:
df_han = han_pipeline.get_session_table(if_load_bpod=True)
df_han.columns

In [ ]:
df_han[(df_han['subject_id'] == '687582') & (df_han['session_date'] == '2023-11-28')].iloc[0]['subject_alias']
# 687582_2023-11-28_12-53-25

In [ ]:
df_han['current_stage_actual'].unique()

In [ ]:
print(df_han.columns.tolist())

### load pre-saved trial df

In [ ]:
# read /data/mice_behavioral_data_20260309-0_200-185844.pkl
with open("/data/mice_behavioral_data_20260309-0_200-185844.pkl", "rb") as f:
    df_0_200 = pickle.load(f)
    
# read /data/mice_behavioral_data_20260309-200_400-114605.pkl
with open("/data/mice_behavioral_data_20260309-200_400-114605.pkl", "rb") as f:
    df_200_400 = pickle.load(f)
    
# read /data/mice_behavioral_data_20260309-400_600-213302.pkl
with open("/data/mice_behavioral_data_20260309-400_600-213302.pkl", "rb") as f:
    df_400_600 = pickle.load(f)

# read /data/mice_behavioral_data_20260310-600_end-120215.pkl
with open("/data/mice_behavioral_data_20260310-600_end-120215.pkl", "rb") as f:
    df_600_end = pickle.load(f)

# read /data/mice_behavioral_data_20260309-misc_778149_778147_753618-215212.pkl
with open("/data/mice_behavioral_data_20260309-misc_778149_778147_753618-215212.pkl", "rb") as f:
    df_misc = pickle.load(f)
    
all_dfs = [df_0_200, df_200_400, df_400_600, df_600_end, df_misc]

In [ ]:
# check if all dfs in all_dfs have the same columns
for idx, df in enumerate(all_dfs):
    print(f"df_{idx}: {len(df)}")
    if not df.columns.to_list().sort() == all_dfs[0].columns.to_list().sort():
        print(f"Columns do not match in {df.shape[0]} vs {all_dfs[0].shape[0]}!")


# concatenate all_dfs into one df
df_all = pd.concat(all_dfs, ignore_index=True)
print(f"df_all: {len(df_all)}")

In [ ]:
df_all.columns

In [ ]:
# check how many unique subjects and sessions are in df_all
print("Total number of subjects:", df_all['subject_id'].nunique())
print("Total number of sessions:", df_all['ses_idx'].nunique())
print("Sessions per subject:")

# sort by number of sessions per subject
print(df_all.groupby('subject_id')['ses_idx'].nunique().sort_values(ascending=False))

In [ ]:
df_mature_session = df_session[
    df_session['current_stage_actual'].isin(['STAGE_FINAL', 'GRADUATED'])
].copy()

In [ ]:
# check how many unique subjects and sessions are in df_mature_session
print("Total number of subjects in df_mature_session:", df_mature_session['subject_id'].nunique())
print("Total number of sessions in df_mature_session:", df_mature_session['ses_idx'].nunique())
print("Sessions per subject in df_mature_session:")
print(df_mature_session.groupby('subject_id')['ses_idx'].nunique().sort_values(ascending=False))

In [ ]:
# generate a session table with subject_id, ses_idx
df_session = df_all[['subject_id', 'ses_idx']].drop_duplicates().reset_index(drop=True)

# get session_date from ses_idx
df_session['session_date'] = df_session['ses_idx'].apply(lambda x: x.split('_')[1])

# based on subject_id and session_date, get nwb_suffix and current_stage_actual from df_han
def get_nwb_suffix_and_current_stage(subject_id, session_date):
    row = df_han[(df_han['subject_id'] == subject_id) & (df_han['session_date'] == session_date)]
    if len(row) == 0:
        print(f"Warning: no row found for subject_id {subject_id} and session_date {session_date} in df_han!")
        return None, None, None
    elif len(row) > 1:
        print(f"Warning: multiple rows found for subject_id {subject_id} and session_date {session_date} in df_han!")
        return None, None, None
    else:
        return row.iloc[0]['nwb_suffix'], row.iloc[0]['current_stage_actual'], row.iloc[0]['curriculum_name']

df_session[['nwb_suffix', 'current_stage_actual', 'curriculum_name']] = df_session.apply(
    lambda x: get_nwb_suffix_and_current_stage(x['subject_id'], x['session_date']),
    axis=1,
    result_type='expand'
)


# check where nwb_suffix is None
print(f"Total number of session not found in df_han: {len(df_session[df_session['nwb_suffix'].isnull()])}")
# df_session[df_session['nwb_suffix'].isnull()]


# drop rows where nwb_suffix is NaN
n_before = len(df_session)
df_session = df_session.dropna(subset=['nwb_suffix']).reset_index(drop=True)
n_after = len(df_session)
print(f"Dropped {n_before - n_after} rows with NaN nwb_suffix. Remaining rows: {n_after}")

In [ ]:
# print number of sessions per subject, sorted
print("Number of sessions per subject (sorted):")
print(df_session.groupby('subject_id').size().sort_values(ascending=False))

In [ ]:
print("Number of mature sessions per subject (sorted):")
print(df_mature_session.groupby('subject_id').size().sort_values(ascending=False))

In [ ]:
# print number of sessions per curriculum, sorted
df_session['curriculum_name'].value_counts().sort_values(ascending=False)

In [ ]:
# print number of sessions per curriculum, sorted
df_mature_session['curriculum_name'].value_counts().sort_values(ascending=False)

In [ ]:
# First group by curriculum_name, then group by subject_id, and print number of sessions per subject, sorted
(df_session
    .groupby(['curriculum_name', 'subject_id'])
    .size()
    .rename('n_sessions')
    .reset_index()
    .sort_values(['curriculum_name', 'n_sessions'], ascending=[True, False])
    .set_index(['curriculum_name', 'subject_id'])
)

In [ ]:
# print each curriculum separately, the subjects in that curriculum, and the number of sessions for each subject, sorted by number of sessions
curriculums = df_session['curriculum_name'].unique()
for curriculum in curriculums:
    print(f"Curriculum: {curriculum}")
    df_curriculum = df_session[df_session['curriculum_name'] == curriculum]
    print(df_curriculum.groupby('subject_id').size().rename('n_sessions').sort_values(ascending=False))
    print("\n")

In [ ]:
(df_mature_session
    .groupby(['curriculum_name', 'subject_id'])
    .size()
    .rename('n_sessions')
    .reset_index()
    .sort_values(['curriculum_name', 'n_sessions'], ascending=[True, False])
    .set_index(['curriculum_name', 'subject_id'])
)

In [ ]:
# print each curriculum separately, the subjects in that curriculum, and the number of sessions for each subject, sorted by number of sessions
curriculums = df_mature_session['curriculum_name'].unique()
for curriculum in curriculums:
    print(f"Curriculum: {curriculum}")
    df_curriculum = df_mature_session[df_mature_session['curriculum_name'] == curriculum]
    print(df_curriculum.groupby('subject_id').size().rename('n_sessions').sort_values(ascending=False))
    print("\n")

In [ ]:
# check how many subjects contain more than 10 mature_sessions
n_session_threshold = 10
mature_sessions_per_subject = df_mature_session.groupby('subject_id').size()
n_subjects_enough = (mature_sessions_per_subject > n_session_threshold).sum()

print(f"Number of subjects with >10 mature sessions: {n_subjects_enough}")
print("Subject IDs with >10 mature sessions:")
print(mature_sessions_per_subject[mature_sessions_per_subject > n_session_threshold].sort_values(ascending=False))

In [ ]:
# get the list of subject_ids of the top k subjects with most mature sessions
# per curriculum in ['Uncoupled Without Baiting', 'Uncoupled Baiting', 'Coupled Baiting']

curricula = ['Uncoupled Without Baiting', 'Uncoupled Baiting', 'Coupled Baiting']
k = 6


top_k_per_curriculum = (
    df_mature_session[df_mature_session['curriculum_name'].isin(curricula)]
    .groupby(['curriculum_name', 'subject_id'])
    .size()
    .rename('n_mature_sessions')
    .reset_index()
    .sort_values(['curriculum_name', 'n_mature_sessions'], ascending=[True, False])
    .groupby('curriculum_name')
    .head(k)
    .set_index(['curriculum_name', 'subject_id'])
)

print(top_k_per_curriculum)

top_k_subject_ids = (
    top_k_per_curriculum
    .reset_index()
    .groupby('curriculum_name')['subject_id']
    .apply(list)
)
print("\nTop " + str(k) + " subject_ids per curriculum:")
print(top_k_subject_ids)


In [ ]:
# flatten the list of subject_ids in top_k_subject_ids into a single list of unique subject_ids
train_set_ids = []
test_set_ids = []
for subject_list in top_k_subject_ids:
    train_set_ids.extend(subject_list[:k//2])
    test_set_ids.extend(subject_list[k//2:k])

print("Unique subject_ids in top " + str(k) + " per curriculum:")
print("Train set subject_ids:", train_set_ids)
print("Test set subject_ids:", test_set_ids)

In [ ]:
# select rows in df_all where subject_id is in train_set_ids or test_set_ids
# retain only the selected columns

cols_to_retain = ['trial', 'subject_id', 'ses_idx', 'animal_response', 'earned_reward']

df_train = df_all[df_all['subject_id'].isin(train_set_ids)][cols_to_retain].copy()
df_test = df_all[df_all['subject_id'].isin(test_set_ids)][cols_to_retain].copy()

In [ ]:
def plot_session_distribution(
    df,
    subject_col='subject_id',
    figsize=(12, 4),
    dpi=150,
    xtick_step=5,
    title_suffix=''
 ):
    sessions_per_subject = df.groupby(subject_col).size()
    if sessions_per_subject.empty:
        raise ValueError('No rows available to plot session distribution.')

    n_sessions_sorted = sessions_per_subject.sort_values(ascending=False)
    x_min, x_max = sessions_per_subject.min(), sessions_per_subject.max()
    bins = np.arange(x_min, x_max + 2) - 0.5

    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=figsize, dpi=dpi)

    # left: histogram of sessions per subject
    axs[0].hist(sessions_per_subject, bins=bins, edgecolor='black', alpha=0.8)
    axs[0].set_xlabel('n_sessions')
    axs[0].set_ylabel('Number of subjects')
    axs[0].set_title(f'Distribution of n_sessions per subject{title_suffix}')
    axs[0].set_xticks(np.arange(x_min, x_max + 1, xtick_step))

    # right: cumulative total sessions while adding subjects from most to least sessions
    cum_sessions = n_sessions_sorted.cumsum()
    subject_rank = np.arange(1, len(cum_sessions) + 1)
    axs[1].step(subject_rank, cum_sessions.values, where='post')
    axs[1].set_xlabel('Number of subjects included (ranked by n_sessions, desc)')
    axs[1].set_ylabel('Cumulative number of sessions')
    axs[1].set_yscale('log')
    axs[1].set_title(f'Cumulative sessions by subject rank{title_suffix}')
    axs[1].set_xlim(1, len(cum_sessions))
    axs[1].grid(alpha=0.3)

    fig.tight_layout()
    return fig, axs

In [ ]:
fig, axs = plot_session_distribution(df_session)
plt.show()

In [ ]:
fig, axs = plot_session_distribution(df_mature_session, title_suffix='mature_only')
plt.show()